# Notebook 1: Linear and Logistic Regression on Composite Microstructures

**UKACM Autumn School: AI for Computational Mechanics**

### The problem

A unidirectional fibre composite is described by its microstructure: where the fibres sit in the
cross-section. Finite element homogenisation of that cross-section gives the effective transverse
properties, but each simulation costs time. The question this notebook asks is the simplest possible
version of the surrogate modelling question:

> How much of the effective property can we predict from two numbers describing the microstructure,
> without running the simulation?

The two numbers are **fibre volume fraction** and **fibre diameter**.

### What you will do

1. Fit a straight line by hand, by adjusting the parameters yourself and watching the error.
2. Watch gradient descent find the same line, and watch it fail when the learning rate is wrong.
3. Test whether fibre diameter carries any information at all.
4. Look at the residuals and find the point where a straight line stops being enough.
5. Meet a property that volume fraction cannot predict at all. That property is the reason the rest of the school exists.
6. Turn the regressor into a classifier and use it as a design screen.

### The data

1485 periodic microstructures, generated at six volume fractions (10 to 60%) and three fibre
diameters (8, 10, 12), with 60 to 98 random realisations of each combination. Each was homogenised
to give the transverse elastic and plastic properties:

| Symbol | Meaning |
|---|---|
| `E22`, `E33` | transverse Young's moduli, in the two in-plane directions |
| `G23` | transverse shear modulus |
| `v23`, `v32` | transverse Poisson's ratios |
| `yield22`, `yield33`, `yield23` | yield stresses |

Two combinations of these matter later:

- `E_mean = (E22 + E33)/2` is the **average** transverse stiffness
- `dE = E22 - E33` is the **anisotropy**, how much stiffer the microstructure is in one direction than the other


In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first. Takes a few seconds.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

np.random.seed(0)

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

# Consistent colours used throughout the notebook
C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

print("numpy", np.__version__, "| pandas", pd.__version__)


### Loading the data

Two files are needed:

- `microstructure_labels.csv`, one row per microstructure, with all properties
- `microstructures_96.npz`, the binary images at 96x96, used for the pictures below

The next cell downloads and extracts the dataset automatically.

In [ ]:
# --- Load data ---------------------------------------------------------------
import os, io, zipfile, urllib.request

DATA_URL = "https://raw.githubusercontent.com/CEMS-Lab/autumn-school/main/datasets/machine_learning/NB1_data.zip"
DATA_DIR = "."         # where to look / unpack

def _have():
    return (os.path.exists(os.path.join(DATA_DIR, "microstructure_labels.csv")) and
            os.path.exists(os.path.join(DATA_DIR, "microstructures_96.npz")))

if not _have() and DATA_URL:
    print("Downloading ...")
    with urllib.request.urlopen(DATA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

if not _have():
    try:
        from google.colab import files          # Colab manual upload
        print("Select microstructure_labels.csv and microstructures_96.npz")
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            "Put microstructure_labels.csv and microstructures_96.npz beside the notebook, "
            "or set DATA_URL above.")

df = pd.read_csv(os.path.join(DATA_DIR, "microstructure_labels.csv"))
_npz = np.load(os.path.join(DATA_DIR, "microstructures_96.npz"), allow_pickle=True)
IMAGES, IMG_KEYS = _npz["images"], _npz["keys"]
KEY_TO_IDX = {k: i for i, k in enumerate(IMG_KEYS)}

# Derived targets
df["E_mean"] = (df.E22 + df.E33) / 2          # average transverse stiffness
df["dE"]     =  df.E22 - df.E33               # anisotropy
df["sy_mean"] = (df.yield22 + df.yield33) / 2
df["vf"]     =  df.vol_frac / 100.0           # volume fraction as a fraction, not a percentage

print(f"{len(df)} microstructures   images {IMAGES.shape}")
print(f"volume fractions: {[int(v) for v in sorted(df.vol_frac.unique())]} %")
print(f"diameters:        {[int(v) for v in sorted(df.diameter.unique())]}")
print(f"{int(df.outlier_flag.sum())} rows flagged as outliers (kept for now, see Part 5)")
df.head()


### What the microstructures look like

Black is matrix, white is fibre. The fraction of white pixels is the volume fraction.

In [ ]:
# --- Gallery: one microstructure per volume fraction -------------------------
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
for ax, vfp in zip(axes, sorted(df.vol_frac.unique())):
    row = df[df.vol_frac == vfp].iloc[0]
    ax.imshow(IMAGES[KEY_TO_IDX[row.key]], cmap="gray", interpolation="nearest")
    ax.set_title(f"$V_f$ = {vfp:.0f}%\n$E_{{mean}}$ = {row.E_mean:.2f} GPa", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle("Increasing fibre volume fraction", y=1.06)
plt.tight_layout(); plt.show()


**Browse the dataset.** Move the sliders. At a fixed volume fraction and diameter, every
realisation is a different random arrangement of the same number of fibres. Watch how much the
property changes between realisations that look statistically identical. That scatter is not
measurement noise; it is real, and it comes from the arrangement.

In [ ]:
# --- Interactive microstructure browser --------------------------------------
def browse(volume_fraction=30, diameter=8, realisation=0):
    sub = df[(df.vol_frac == volume_fraction) & (df.diameter == diameter)].reset_index(drop=True)
    if len(sub) == 0:
        print("no samples for this combination"); return
    row = sub.iloc[realisation % len(sub)]

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.6),
                                 gridspec_kw={"width_ratios": [1, 1.5]})
    a1.imshow(IMAGES[KEY_TO_IDX[row.key]], cmap="gray", interpolation="nearest")
    a1.set_title(f"{row.key}", fontsize=8); a1.set_xticks([]); a1.set_yticks([]); a1.grid(False)

    a2.scatter(sub.index, sub.E_mean, s=14, color=C_DATA, alpha=0.6, label="all realisations")
    a2.scatter([realisation % len(sub)], [row.E_mean], s=90, color=C_FIT, zorder=5, label="this one")
    a2.axhline(sub.E_mean.mean(), color="k", ls="--", lw=1, label=f"mean {sub.E_mean.mean():.2f}")
    a2.set_xlabel("realisation index"); a2.set_ylabel("$E_{mean}$  (GPa)")
    a2.set_title(f"{len(sub)} realisations at $V_f$={volume_fraction}%, d={diameter}\n"
                 f"spread: {sub.E_mean.min():.2f} to {sub.E_mean.max():.2f} GPa "
                 f"(std {sub.E_mean.std():.3f})", fontsize=9)
    a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()

interact(browse,
         volume_fraction=IntSlider(30, min=10, max=60, step=10, continuous_update=False),
         diameter=Dropdown(options=[8, 10, 12], value=8),
         realisation=IntSlider(0, min=0, max=97, step=1, continuous_update=False));


---

# Part 1: Fitting a straight line

### The model

The simplest possible surrogate. One input, one output, two parameters:

$$\boxed{\;\hat{E}(V_f) \;=\; w\,V_f + b\;}$$

$V_f$ is the fibre volume fraction as a fraction, so 0.1 to 0.6. $\hat{E}$ is the predicted average
transverse modulus in GPa. $w$ is the slope, in GPa per unit volume fraction, and $b$ is the
intercept, in GPa. Written for the $i$-th microstructure in the dataset,

$$\hat{E}_i = w\,V_{f,i} + b, \qquad i = 1 \ldots N, \qquad N = 1485$$

Both parameters are **learned** from the data. Nothing else about the model is adjustable.

### The loss

We need a single number saying how wrong a given $(w, b)$ is. Mean squared error:

$$L(w, b) \;=\; \frac{1}{N}\sum_{i=1}^{N}\bigl(\hat{E}(V_{f,i}) - E_i\bigr)^2$$

Squared, so that errors above and below the line both count as errors, and so that large errors
count disproportionately. Training means finding the $(w, b)$ that make $L$ as small as possible.

In [ ]:
# --- The model and the loss --------------------------------------------------
x = df["vf"].values          # volume fraction, 0.10 to 0.60
y = df["E_mean"].values      # GPa

def predict(w, b, x=x):
    return w * x + b

def mse(w, b, x=x, y=y):
    return np.mean((predict(w, b, x) - y) ** 2)

print(f"A deliberately bad guess:  w=5,  b=5   ->  L = {mse(5, 5):8.3f}")
print(f"A better guess:            w=17, b=1   ->  L = {mse(17, 1):8.3f}")


### Find the line yourself

The left panel shows the data and your line. The right panel is the **loss landscape**: every point
in it is a different $(w, b)$, coloured by how bad that line is, with your current position marked.

Try to get the marker into the dark centre. Notice the shape of the valley: it is a long thin
ellipse tilted at an angle, not a circular bowl. That tilt means $w$ and $b$ are coupled, and it is
exactly what makes gradient descent slow in the next section.

In [ ]:
# --- Precompute the loss landscape once --------------------------------------
W = np.linspace(0, 34, 180)
B = np.linspace(-10, 14, 180)
WW, BB = np.meshgrid(W, B)
# closed form of the MSE surface, vectorised
LOSS = (np.mean(y**2) - 2*WW*np.mean(x*y) - 2*BB*np.mean(y)
        + WW**2*np.mean(x**2) + 2*WW*BB*np.mean(x) + BB**2)

# The exact minimiser, from the normal equations. Used as a reference marker below.
_A = np.array([[np.mean(x**2), np.mean(x)], [np.mean(x), 1.0]])
w_opt, b_opt = np.linalg.solve(_A, np.array([np.mean(x*y), np.mean(y)]))
L_opt = mse(w_opt, b_opt)
print(f"exact optimum:  w = {w_opt:.4f}   b = {b_opt:.4f}   L = {L_opt:.5f}")

def fit_by_hand(w=10.0, b=0.0):
    L = mse(w, b)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))

    a1.scatter(x, y, s=8, alpha=0.25, color=C_DATA, label="1485 microstructures")
    xs = np.linspace(0.08, 0.62, 50)
    a1.plot(xs, predict(w, b, xs), color=C_FIT, lw=2.5, label=f"$\\hat{{E}}$ = {w:.1f}$V_f$ + {b:.1f}")
    # draw the residuals for a sample of points
    idx = np.arange(0, len(x), 40)
    a1.vlines(x[idx], y[idx], predict(w, b, x[idx]), color=C_BAD, lw=0.7, alpha=0.7)
    a1.set_xlabel("fibre volume fraction  $V_f$"); a1.set_ylabel("$E_{mean}$  (GPa)")
    a1.set_ylim(0, 16); a1.set_title(f"L = {L:.3f}   (red bars are residuals)")
    a1.legend(loc="upper left", fontsize=8)

    cs = a2.contourf(WW, BB, np.log10(LOSS), levels=30, cmap="viridis_r")
    a2.contour(WW, BB, np.log10(LOSS), levels=12, colors="w", linewidths=0.4, alpha=0.5)
    a2.plot(w, b, "o", color=C_FIT, ms=12, mec="w", mew=1.6)
    a2.set_xlabel("slope  $w$"); a2.set_ylabel("intercept  $b$")
    a2.set_title("loss landscape  ($\\log_{10} L$)"); a2.grid(False)
    plt.colorbar(cs, ax=a2, shrink=0.85)
    plt.tight_layout(); plt.show()

interact(fit_by_hand,
         w=FloatSlider(10.0, min=0, max=30, step=0.5, continuous_update=False),
         b=FloatSlider(0.0, min=-6, max=10, step=0.25, continuous_update=False));


### Stop and think

Before running the next cell: at the best line, what should the sum of the red residual bars be?
And can you get the loss to zero by choosing $w$ and $b$ well enough?

---

## Gradient descent

Adjusting sliders does not scale to a model with a million parameters. Instead, use the gradient.
For mean squared error the derivatives are exact:

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i (\hat{E}_i - E_i)\,V_{f,i},
\qquad
\frac{\partial L}{\partial b} = \frac{2}{N}\sum_i (\hat{E}_i - E_i)$$

and the update rule takes a step downhill:

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w},
\qquad
b \leftarrow b - \eta\,\frac{\partial L}{\partial b}$$

$\eta$ is the **learning rate**. It is the single most consequential number in the whole of machine
learning, and the animation below shows why.

In [ ]:
# --- Gradient descent --------------------------------------------------------
def gradient_descent(lr, n_steps=150, w0=8.0, b0=6.0):
    w, b = w0, b0
    path = [(w, b, mse(w, b))]
    for _ in range(n_steps):
        err = predict(w, b) - y
        gw  = 2 * np.mean(err * x)
        gb  = 2 * np.mean(err)
        w  -= lr * gw
        b  -= lr * gb
        if not np.isfinite(w) or abs(w) > 1e4:      # diverged
            path.append((np.nan, np.nan, np.nan)); break
        path.append((w, b, mse(w, b)))
    return np.array(path)

# Stability limit for this problem. For mean squared error the Hessian is constant,
# and gradient descent is stable only while eta < 2 / (largest eigenvalue).
H = 2 * np.array([[np.mean(x**2), np.mean(x)], [np.mean(x), 1.0]])
eta_max = 2 / np.linalg.eigvalsh(H).max()
print(f"largest Hessian eigenvalue = {np.linalg.eigvalsh(H).max():.4f}")
print(f"gradient descent is stable only for eta < {eta_max:.4f}\n")

for lr in [0.05, 0.5, 0.91]:
    p = gradient_descent(lr)
    end = p[-1]
    status = "diverged" if not np.isfinite(end[2]) else f"L = {end[2]:.4f}"
    print(f"eta = {lr:4}   after {len(p)-1:3d} steps:  {status}")


### Watch it learn

Three learning rates, same starting point, same data.

- **Too small**: correct direction, not enough steps to arrive.
- **About right**: converges.
- **Too large**: each step overshoots the valley floor and lands further up the opposite wall.

The animation takes a few seconds to build. Use the play button.

In [ ]:
# --- Animated gradient descent (three learning rates) ------------------------
# Window for the landscape panel, centred on the optimum.
WLO, WHI, BLO, BHI = 2, 32, -9, 12

lrs    = [0.05, 0.5, 0.91]
labels = ["too small", "about right", "too large"]
cols   = [C_ALT, C_FIT, C_BAD]
paths  = [gradient_descent(lr) for lr in lrs]

def steps_in_window(p):
    # How many leading steps stay inside the landscape panel.
    ok = ((p[:, 0] >= WLO) & (p[:, 0] <= WHI) &
          (p[:, 1] >= BLO) & (p[:, 1] <= BHI) & np.isfinite(p[:, 0]))
    return int(np.argmax(~ok)) if (~ok).any() else len(p)

vis = [steps_in_window(p) for p in paths]
for lr, p, v in zip(lrs, paths, vis):
    where = "stays in view throughout" if v >= len(p) else f"leaves the landscape view at step {v}"
    print(f"eta = {lr:4}: {len(p)-1:3d} steps computed, {where}")

n_fr   = 60
n_step = max(len(p) for p in paths)
frames = np.linspace(0, n_step - 1, n_fr).astype(int)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13.5, 4.2))
xs = np.linspace(0.08, 0.62, 50)

# --- panel 1: the fit -------------------------------------------------------
ax1.scatter(x, y, s=6, alpha=0.2, color=C_DATA)
lines = [ax1.plot([], [], lw=2.2, color=c, label=f"$\\eta$={lr}  ({lab})")[0]
         for c, lr, lab in zip(cols, lrs, labels)]
ax1.set_xlim(0.08, 0.62); ax1.set_ylim(0, 16)
ax1.set_xlabel("$V_f$"); ax1.set_ylabel("$E_{mean}$ (GPa)")
ax1.legend(fontsize=8, loc="upper left"); ax1.set_title("the fit")

# --- panel 2: the loss landscape -------------------------------------------
ax2.contourf(WW, BB, np.log10(LOSS), levels=30, cmap="viridis_r")
ax2.contour(WW, BB, np.log10(LOSS), levels=12, colors="w", linewidths=0.4, alpha=0.5)
ax2.plot(w_opt, b_opt, marker="*", ms=16, color="w", mec="k", mew=1.0, zorder=2, ls="none")
# Diverging runs are drawn underneath and only while they are inside the window,
# so they cannot scribble over the runs that matter.
# The diverging run oscillates across the valley many times before it leaves the
# window. Drawing all of it turns the panel into a scribble, so it is cut off after
# a few overshoots and the rest of its story is told by the loss history on the right.
DIV_SHOW = 10
cap    = [len(paths[0]), len(paths[1]), DIV_SHOW]
zord   = [7, 7, 2]
widths = [2.0, 2.0, 1.0]
alphas = [1.0, 1.0, 0.55]
styles = ["-o", "-o", "--o"]
trails = [ax2.plot([], [], st, ms=3.5, lw=lw, color=c, alpha=al, zorder=z)[0]
          for c, st, lw, al, z in zip(cols, styles, widths, alphas, zord)]
heads  = [ax2.plot([], [], "o", ms=9, color=c, mec="w", mew=1.4, zorder=z + 1)[0]
          for c, z in zip(cols, zord)]
note = ax2.text(0.03, 0.965, "", transform=ax2.transAxes, ha="left", va="top",
                fontsize=8, color=C_BAD,
                bbox=dict(fc="w", ec=C_BAD, alpha=0.92, boxstyle="round,pad=0.28"))
ax2.set_xlim(WLO, WHI); ax2.set_ylim(BLO, BHI); ax2.grid(False)
ax2.set_xlabel("$w$"); ax2.set_ylabel("$b$")
ax2.set_title("path across the loss landscape\n(star is the exact optimum)", fontsize=9.5)

# --- panel 3: the loss history ---------------------------------------------
curves = [ax3.plot([], [], lw=2, color=c)[0] for c in cols]
ax3.axhline(L_opt, color="k", ls=":", lw=1.2)
ax3.text(n_step * 0.98, L_opt * 1.25, "lowest possible loss",
         ha="right", va="bottom", fontsize=8)
ax3.set_xlim(0, n_step); ax3.set_yscale("log"); ax3.set_ylim(2e-1, 1e4)
ax3.set_xlabel("step"); ax3.set_ylabel("loss  $L$"); ax3.set_title("loss history")

def update(f):
    k = frames[f]
    msg = ""
    for i, p in enumerate(paths):
        j     = min(k, len(p) - 1)
        limit = min(vis[i], cap[i])                 # window clip and scribble clip
        jv    = min(j, max(limit - 1, 0))
        w, b, _ = p[jv]
        lines[i].set_data(xs, w * xs + b)
        trails[i].set_data(p[:jv+1, 0], p[:jv+1, 1])
        heads[i].set_data([p[jv, 0]], [p[jv, 1]])
        hist = p[:j+1, 2]
        curves[i].set_data(np.arange(len(hist)), np.clip(hist, None, 5e3))
        if limit < len(p) and k >= limit:
            msg = (f"$\\eta$={lrs[i]} keeps overshooting.\n"
                   f"Only its first {DIV_SHOW} steps are drawn;\n"
                   f"it leaves this view at step {vis[i]}.")
    note.set_text(msg)
    return lines + trails + heads + curves + [note]

anim = animation.FuncAnimation(fig, update, frames=n_fr, interval=110, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


### What the animation shows

Three things, in the order they become visible.

The **valley is a long thin ellipse**, not a circular bowl. Gradient descent steps perpendicular to
the contours, so it crosses the valley quickly and creeps along it slowly. That is why the green
path moves fast at first and then almost stops.

The **step size sets the behaviour, not the direction**. All three runs compute the same gradient at
the same starting point. Only the multiplier differs.

**Divergence is not gradual.** For mean squared error the Hessian is constant, so there is an exact
threshold: $\eta < 2/\lambda_{max}$, which the cell above printed. Below it the loss falls. Above
it each step lands further from the minimum than the last, and the run leaves the picture
altogether. Nothing warns you except the loss going up.

### The answer gradient descent was looking for

For linear regression the minimum can be written down directly, so we can check.

In [ ]:
# --- Closed form, and the sklearn equivalent ---------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

lin = LinearRegression().fit(x.reshape(-1, 1), y)
w_star, b_star = lin.coef_[0], lin.intercept_
y_hat = lin.predict(x.reshape(-1, 1))

gd = gradient_descent(0.5, n_steps=2000)[-1]

print(f"gradient descent (2000 steps):  w = {gd[0]:7.4f}   b = {gd[1]:7.4f}   L = {gd[2]:.5f}")
print(f"closed form:                    w = {w_star:7.4f}   b = {b_star:7.4f}   L = {mse(w_star, b_star):.5f}")
print()
print(f"R2  = {r2_score(y, y_hat):.4f}      MAE = {mean_absolute_error(y, y_hat):.4f} GPa")
print()
print("The fitted model is")
print(f"    E_mean(Vf) = {w_star:.4f} * Vf + {b_star:.4f}      [GPa]")
print()
print(f"Reading the slope as physics: every 10 percentage points of fibre")
print(f"adds about {w_star * 0.10:.2f} GPa of average transverse stiffness.")


$R^2 = 0.95$ from a single input and two parameters. Most of the average transverse stiffness of
this composite is determined by how much fibre there is, which is the answer a mechanician would
have predicted. The interesting question is what the remaining 5% is made of.

### One thing to be clear about before going further

Every $R^2$ printed so far is measured on the same 1485 rows the model was fitted to, outliers
included. That is an in-sample fit, not a test score, and in general an in-sample score flatters the
model. Here it does not, because two parameters cannot memorise 1485 points, but the habit of
checking is the point. The next cell refits on three quarters of the data and scores on the quarter
it never saw.

In [ ]:
# --- In-sample against held out ----------------------------------------------
from sklearn.model_selection import train_test_split as _tts

_c = df[~df.outlier_flag]                       # drop the 15 flagged outliers, as Notebooks 2 and 3 do
_Xa, _ya = _c[["vf"]].values, _c["E_mean"].values
_Xtr, _Xte, _ytr, _yte = _tts(_Xa, _ya, test_size=0.25, random_state=0)

_Xall  = df[["vf"]].values                                  # all 1485 rows, outliers kept
_m_in  = LinearRegression().fit(_Xall, y)
_m_out = LinearRegression().fit(_Xtr, _ytr)                 # 75% of the 1470 clean rows

_Xq  = np.c_[df.vf, df.vf**2]
_Xqtr, _Xqte = np.c_[_Xtr[:, 0], _Xtr[:, 0]**2], np.c_[_Xte[:, 0], _Xte[:, 0]**2]
_q_in  = LinearRegression().fit(_Xq, y)
_q_out = LinearRegression().fit(_Xqtr, _ytr)

print(f"{'model':16s} {'in-sample R2':>14s} {'held-out test R2':>18s}")
print(f"{'linear':16s} {r2_score(y, _m_in.predict(_Xall)):14.4f} "
      f"{r2_score(_yte, _m_out.predict(_Xte)):18.4f}")
print(f"{'quadratic':16s} {r2_score(y, _q_in.predict(_Xq)):14.4f} "
      f"{r2_score(_yte, _q_out.predict(_Xqte)):18.4f}")
print()
print(f"in-sample fit:  all {len(df)} rows, outliers kept")
print(f"held-out test:  {len(_Xtr)} train and {len(_Xte)} test of the {len(_c)} rows that remain "
      f"after dropping outliers")
print("Notebooks 2 and 3 use that same 75/25 split, so their numbers are comparable to the right column.")


The two columns agree to within 0.001, and the held-out score is very slightly the higher of
the two. A two-parameter model has nothing to overfit with, so the in-sample number happens to be
honest here. It will not be once the models have
thousands of parameters, which is why every score from Part 5 onwards, and every score in Notebooks
2 and 3, is measured on data the model never saw.

---

# Part 2: Does fibre diameter matter?

The dataset varies fibre diameter over 8, 10 and 12 at every volume fraction. It is a real input,
it was deliberately varied, and it is sitting in the table. The obvious move is to add it to the
model.

Before doing that, a prediction. For a periodic microstructure with perfectly bonded phases, the
effective properties depend on the volume fraction and the arrangement, not on the absolute size of
the fibres. Scale the whole geometry up and nothing changes. So diameter *should* contribute
nothing.

Test it.

In [ ]:
# --- With and without diameter -----------------------------------------------
X1 = df[["vf"]].values
X2 = df[["vf", "diameter"]].values

m1 = LinearRegression().fit(X1, y)
m2 = LinearRegression().fit(X2, y)

print(f"{'model':28s} {'R2':>8s}   coefficients")
print(f"{'E_mean ~ Vf':28s} {r2_score(y, m1.predict(X1)):8.5f}   Vf: {m1.coef_[0]:+.4f}")
print(f"{'E_mean ~ Vf + diameter':28s} {r2_score(y, m2.predict(X2)):8.5f}   "
      f"Vf: {m2.coef_[0]:+.4f}   diameter: {m2.coef_[1]:+.5f}")
print()
print(f"R2 gained by adding diameter: {r2_score(y, m2.predict(X2)) - r2_score(y, m1.predict(X1)):.6f}")


In [ ]:
# --- Is the diameter effect bigger than the realisation-to-realisation scatter?
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))

for d, mk in zip([8, 10, 12], ["o", "s", "^"]):
    s = df[df.diameter == d]
    a1.scatter(s.vf, s.E_mean, s=10, alpha=0.45, marker=mk, label=f"d = {d}")
a1.set_xlabel("$V_f$"); a1.set_ylabel("$E_{mean}$ (GPa)")
a1.set_title("coloured by diameter: the three groups overlap"); a1.legend(fontsize=8)

g = df.groupby(["vol_frac", "diameter"]).E_mean
cell_mean, cell_std = g.mean().unstack(), g.std().unstack()
rng = (cell_mean.max(axis=1) - cell_mean.min(axis=1))
scatter = cell_std.mean(axis=1)
xp = np.arange(len(rng)); wdt = 0.38
a2.bar(xp - wdt/2, rng.values,     wdt, color=C_FIT,  label="spread between diameters")
a2.bar(xp + wdt/2, scatter.values, wdt, color=C_DATA, label="scatter between realisations")
a2.set_xticks(xp); a2.set_xticklabels([f"{v:.0f}%" for v in rng.index])
a2.set_xlabel("volume fraction"); a2.set_ylabel("$E_{mean}$ spread (GPa)")
a2.set_title("the diameter effect is the same size as random scatter"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()


**What the two panels show.** Left: the three diameter groups lie on top of one another, so
knowing the diameter does not help you place a microstructure on the stiffness axis. Right: the
orange bar is how far apart the diameter groups are, the blue bar is how much two random
realisations of a *single* diameter differ. Blue is as large as orange at every volume fraction. An
effect smaller than the noise you already have is an effect you cannot use.

The fitted coefficient on diameter is about $-0.016$ GPa per unit diameter, printed by the cell
above, and adding diameter changes $R^2$ in the fifth decimal place. At every volume fraction the spread between diameter groups is
comparable to the spread between random realisations of a single diameter.

Diameter is a **null feature**. The model found that on its own, from the data, and it agrees with
what homogenisation theory says should happen.

This is worth dwelling on. A feature being present in your dataset, and having been varied
deliberately by whoever generated it, is not evidence that it matters. The fitted coefficient is
evidence.

One caveat, which is Exercise 4. "No effect on the mean" is not the same as "no effect at all".
Look again at the orange bars above: they grow with volume fraction. Diameter turns out to change
something, just not the thing we asked the model to predict.

---

# Part 3: Where the straight line runs out

$R^2 = 0.95$ sounds close to finished. Residual plots are how you find out whether it is.

A residual is the part of the measurement the model failed to account for:

$$r_i \;=\; E_i - \hat{E}_i$$

If the model has captured the structure in the data, the residuals should look like noise: no
pattern, no trend, centred on zero at every value of the input. If they curve, the model is missing
something systematic.

### The two models being compared

**Linear**, the one fitted in Part 1, with two parameters:

$$\hat{E}(V_f) = w_1 V_f + b$$

**Quadratic**, one extra term and one extra parameter:

$$\hat{E}(V_f) = w_1 V_f + w_2 V_f^2 + b$$

Both are still *linear regression*, because both are linear in the parameters $w_1, w_2, b$. The
model does not become nonlinear just because a squared input appears in it. All that has changed is
the set of features handed to the fit: $[V_f]$ becomes $[V_f,\; V_f^2]$. The normal equations solve
both in exactly the same way. This distinction matters, because it is the reason a great deal can be
achieved with linear regression and well-chosen features, which is the subject of Notebook 2.

In [ ]:
# --- Residuals -----------------------------------------------------------------
resid = y - m1.predict(X1)

# quadratic in Vf
Xq = np.c_[df.vf, df.vf**2]
mq = LinearRegression().fit(Xq, y)
resid_q = y - mq.predict(Xq)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

axes[0].scatter(x, resid, s=8, alpha=0.3, color=C_DATA)
for vfp in sorted(df.vol_frac.unique()):
    m = df.vol_frac == vfp
    axes[0].plot(df.vf[m].iloc[0], resid[m].mean(), "o", color=C_BAD, ms=9, zorder=5)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel("$V_f$"); axes[0].set_ylabel("residual (GPa)")
axes[0].set_title("linear model: residual means curve\n(red = mean residual per $V_f$)", fontsize=9)

axes[1].scatter(x, resid_q, s=8, alpha=0.3, color=C_ALT)
for vfp in sorted(df.vol_frac.unique()):
    m = df.vol_frac == vfp
    axes[1].plot(df.vf[m].iloc[0], resid_q[m].mean(), "o", color=C_BAD, ms=9, zorder=5)
axes[1].axhline(0, color="k", lw=1)
axes[1].set_ylim(axes[0].get_ylim())
axes[1].set_xlabel("$V_f$"); axes[1].set_ylabel("residual (GPa)")
axes[1].set_title("add $V_f^2$: the curvature is gone", fontsize=9)

xs = np.linspace(0.08, 0.62, 100)
axes[2].scatter(x, y, s=7, alpha=0.2, color=C_DATA)
axes[2].plot(xs, m1.predict(xs.reshape(-1, 1)), color=C_FIT, lw=2,
             label=f"linear, $R^2$={r2_score(y, m1.predict(X1)):.4f}")
axes[2].plot(xs, mq.predict(np.c_[xs, xs**2]), color=C_ALT, lw=2, ls="--",
             label=f"quadratic, $R^2$={r2_score(y, mq.predict(Xq)):.4f}")
axes[2].set_xlabel("$V_f$"); axes[2].set_ylabel("$E_{mean}$ (GPa)")
axes[2].legend(fontsize=8); axes[2].set_title("the two fits", fontsize=9)
plt.tight_layout(); plt.show()


**What the three panels show.** Left and middle are the same residuals plotted against the same
axis, for the linear and the quadratic model, on a shared vertical scale so they can be compared
directly. The red markers are the mean residual at each volume fraction, which is the quickest way
to see a trend through a cloud of points. Right is the two fitted curves over the data.

The mean residual of the linear model is positive at the ends and negative in the middle. That
is a curve, not noise. Stiffness rises faster than linearly with volume fraction, which is what
micromechanics predicts for transverse loading: the fibres start to interact as they crowd together.

Adding one quadratic term takes $R^2$ from 0.95 to 0.99 and flattens the residuals.

This is the first argument for nonlinear models. It is a weak argument, because here we could see
the right term by eye and add it by hand. The next part is the strong argument.

---

# Part 4: A property that volume fraction cannot predict

So far the target has been $E_{mean}$, the average of the two transverse moduli. Now take the
difference:

$$\Delta E = E_{22} - E_{33}$$

This is the **anisotropy** of the cross-section: how much stiffer the microstructure is in one
transverse direction than the other. It matters. A composite that is meant to be transversely
isotropic but is not will not behave as designed.

Both microstructures below have the same volume fraction and the same fibre diameter. One is
noticeably anisotropic and one is not. Nothing about the composition distinguishes them. Only the
arrangement does.

In [ ]:
# --- Two microstructures, same composition, different anisotropy -------------
cell = df[(df.vol_frac == 40) & (df.diameter == 8)].copy()
lo = cell.loc[cell.dE.abs().idxmin()]
hi = cell.loc[cell.dE.abs().idxmax()]

fig, axes = plt.subplots(1, 2, figsize=(7.5, 4))
for ax, r, lab in zip(axes, [lo, hi], ["nearly isotropic", "clearly anisotropic"]):
    ax.imshow(IMAGES[KEY_TO_IDX[r.key]], cmap="gray", interpolation="nearest")
    ax.set_title(f"{lab}\n$V_f$=40%, d=8\n$E_{{22}}$={r.E22:.2f}, $E_{{33}}$={r.E33:.2f}, "
                 f"$\\Delta E$={r.dE:+.2f} GPa", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()


In [ ]:
# --- Try to predict dE from composition --------------------------------------
targets = {"E_mean": "average stiffness", "dE": "anisotropy",
           "sy_mean": "average yield stress"}

print(f"{'target':10s} {'':22s} {'R2 from Vf':>12s} {'R2 from Vf + Vf^2 + d':>23s}")
print("-" * 70)
results = {}
for t, desc in targets.items():
    yy = df[t].values
    r_lin = r2_score(yy, LinearRegression().fit(X1, yy).predict(X1))
    Xf = np.c_[df.vf, df.vf**2, df.diameter]
    r_ric = r2_score(yy, LinearRegression().fit(Xf, yy).predict(Xf))
    results[t] = (r_lin, r_ric)
    print(f"{t:10s} {desc:22s} {r_lin:12.4f} {r_ric:23.4f}")


In [ ]:
# --- The picture that makes the point ----------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for ax, t, ttl in zip(axes, ["E_mean", "dE"],
                      ["$E_{mean}$: composition explains it",
                       "$\\Delta E$: composition explains nothing"]):
    yy = df[t].values
    mm = LinearRegression().fit(X1, yy)
    ax.scatter(x, yy, s=9, alpha=0.3, color=C_DATA)
    xs = np.linspace(0.08, 0.62, 50)
    ax.plot(xs, mm.predict(xs.reshape(-1, 1)), color=C_FIT, lw=2.5)
    ax.set_xlabel("$V_f$"); ax.set_ylabel(f"{t}  (GPa)")
    ax.set_title(f"{ttl}\n$R^2$ = {results[t][0]:.4f}", fontsize=10)
plt.tight_layout(); plt.show()

print(f"Anisotropy ranges from {df.dE.min():+.2f} to {df.dE.max():+.2f} GPa.")
print(f"Volume fraction and diameter together explain {results['dE'][1]*100:.2f}% of it.")


**What the two panels show.** Identical axes, identical data, identical model. The only thing
that changed is which quantity is on the vertical axis. On the left the fitted line passes through
the middle of a clear trend. On the right the data is a horizontal band and the fitted line is flat,
because there is no trend for it to find.

$R^2 = 0.0005$ with volume fraction, $V_f^2$ and diameter together, as the table above prints.
Not weak. Nothing.

$\Delta E$ varies from $-1.77$ to $+1.55$ GPa across the dataset, so there is plenty of signal to
predict. Composition simply carries none of it. The information about anisotropy lives entirely in
where the fibres are, and we threw that away the moment we reduced a 96x96 image to two numbers.

This is the reason for the rest of the school:

| How the microstructure is represented | What it can predict |
|---|---|
| Two numbers: $V_f$, diameter | $E_{mean}$ well, $\Delta E$ not at all |
| Engineered descriptors: spacing, local density, pair correlations (**Notebook 2**) | $\Delta E$ substantially |
| The raw image, through a CNN (**Notebook 3**) | $\Delta E$ better still |

Notice what this is not. It is not an argument that neural networks are better than regression. It
is an argument that **the representation of the input decides the ceiling**, and no amount of model
capacity gets past a representation that has already discarded the answer.

---

# Part 5: From regression to classification

A design question is often not "what is the stiffness" but "does this microstructure meet the
requirement". That is a classification problem.

Take a design target of $E_{mean} \ge 8$ GPa and label each microstructure pass or fail.

### The model

Start with the same linear function as before, and call its output the **logit**:

$$z(V_f) \;=\; w\,V_f + b$$

$z$ can be any real number, so it cannot be a probability. Pass it through the **logistic sigmoid**:

$$\sigma(z) \;=\; \frac{1}{1 + e^{-z}}$$

which maps the whole real line into the open interval $(0,1)$. The model is the composition of the
two:

$$\boxed{\;P(\text{pass} \mid V_f) \;=\; \sigma\!\left(w V_f + b\right) \;=\; \frac{1}{1 + e^{-(w V_f + b)}}\;}$$

Three properties of $\sigma$ are worth knowing before the next cell.

$\sigma(0) = 1/2$, so the decision boundary sits where $z = 0$, that is at $V_f = -b/w$.

$w$ controls how sharply the probability switches from 0 to 1. Large $|w|$ gives a steep transition
and a confident model, small $|w|$ a gradual one.

Its derivative is $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$, which is largest at $z=0$ and tends to
zero at both ends. That is convenient for training, and it is also why sigmoid activations saturate
in deep networks, which comes up in Notebook 2.

### Why not just threshold the regressor?

You can, and often should. But this model returns a probability rather than a hard yes or no, and
that probability is what lets you choose how cautious to be. The cells after next are about that
choice.

Note the structure: a linear function, then a nonlinear squashing function. That is exactly one
neuron. Notebook 2 starts from here.

In [ ]:
# --- The sigmoid, and what w and b do to it ----------------------------------
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

zs = np.linspace(-8, 8, 400)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))

a1.plot(zs, sigmoid(zs), color=C_DATA, lw=2.5, label=r"$\sigma(z)$")
a1.plot(zs, sigmoid(zs)*(1-sigmoid(zs)), color=C_ALT, lw=2, ls="--",
        label=r"$\sigma'(z)=\sigma(1-\sigma)$")
a1.axhline(0.5, color="k", lw=0.9, ls=":"); a1.axvline(0, color="k", lw=0.9, ls=":")
a1.plot(0, 0.5, "o", color=C_BAD, ms=8, zorder=5)
a1.annotate("$\\sigma(0)=1/2$", xy=(0, 0.5), xytext=(1.6, 0.33), fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.9))
a1.set_xlabel("$z$"); a1.set_ylabel("")
a1.set_title("the logistic sigmoid and its derivative", fontsize=10)
a1.legend(fontsize=8, loc="upper left")

vfs = np.linspace(0, 0.8, 400)
for w_demo, ls in [(5, ":"), (15, "-"), (40, "--")]:
    b_demo = -w_demo * 0.35                       # boundary pinned at Vf = 0.35
    a2.plot(vfs, sigmoid(w_demo*vfs + b_demo), lw=2, ls=ls, label=f"$w$ = {w_demo}")
a2.axvline(0.35, color="k", lw=1, ls=":")
a2.text(0.355, 0.05, "$V_f=-b/w$", fontsize=9)
a2.axhline(0.5, color="k", lw=0.8, ls=":")
a2.set_xlabel("$V_f$"); a2.set_ylabel("P(pass)")
a2.set_title("same boundary, three values of $w$", fontsize=10); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()


$b$ moves the transition left or right, $w$ makes it sharper or softer. A model with a large
$w$ is confident everywhere except in a narrow band. A model with a small $w$ hedges over a wide
range of volume fractions. Neither is automatically better, and the fit chooses both from the data.

In [ ]:
# --- Clean the data, then build the classification target --------------------
clean = df[~df.outlier_flag].copy()
print(f"Dropped {int(df.outlier_flag.sum())} flagged outliers, {len(clean)} rows remain.")

THRESHOLD_GPA = 8.0
clean["passes"] = (clean.E_mean >= THRESHOLD_GPA).astype(int)
print(f"Design target E_mean >= {THRESHOLD_GPA} GPa:  "
      f"{clean.passes.sum()} pass, {(1-clean.passes).sum()} fail")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score

Xc = clean[["vf", "diameter"]].values
yc = clean["passes"].values
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)

clf = LogisticRegression().fit(Xtr, ytr)
p_te = clf.predict_proba(Xte)[:, 1]
print(f"\nTest accuracy at the default 0.5 cutoff: {accuracy_score(yte, p_te >= 0.5):.3f}")


### The cutoff is a design decision, not a default

0.5 is a convention, not an answer. Move the slider.

Two kinds of mistake are possible and they are not equally bad. A **false pass** is a microstructure
the model clears that actually falls short of the requirement, which reaches the component. A
**false fail** is a perfectly good design thrown away. Which one you can tolerate depends on the
application, and the cutoff is where you encode that judgement.

In [ ]:
# --- Interactive cutoff -------------------------------------------------------
def cutoff_explorer(cutoff=0.5):
    pred = (p_te >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    acc  = (tp + tn) / len(yte)
    prec = tp / (tp + fp) if (tp + fp) else np.nan
    rec  = tp / (tp + fn) if (tp + fn) else np.nan

    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13.5, 3.8))

    cm = np.array([[tn, fp], [fn, tp]])
    a1.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
    for i in range(2):
        for j in range(2):
            a1.text(j, i, cm[i, j], ha="center", va="center", fontsize=15,
                    color="w" if cm[i, j] > cm.max()/2 else "k")
    a1.set_xticks([0, 1], ["predicted\nfail", "predicted\npass"])
    a1.set_yticks([0, 1], ["actually\nfail", "actually\npass"])
    a1.set_title(f"{fp} false passes, {fn} false fails", fontsize=10); a1.grid(False)

    a2.hist(p_te[yte == 0], bins=25, alpha=0.65, color=C_BAD,  label="actually fails")
    a2.hist(p_te[yte == 1], bins=25, alpha=0.65, color=C_ALT,  label="actually passes")
    a2.axvline(cutoff, color="k", lw=2.2, ls="--")
    a2.set_xlabel("predicted probability of passing"); a2.set_ylabel("count")
    a2.set_title(f"accuracy {acc:.3f}   precision {prec:.3f}   recall {rec:.3f}", fontsize=9)
    a2.legend(fontsize=8)

    fpr, tpr, thr = roc_curve(yte, p_te)
    a3.plot(fpr, tpr, color=C_DATA, lw=2, label=f"AUC = {auc(fpr, tpr):.3f}")
    a3.plot([0, 1], [0, 1], "k--", lw=1)
    k = np.argmin(np.abs(thr - cutoff))
    a3.plot(fpr[k], tpr[k], "o", color=C_FIT, ms=11, mec="w", mew=1.5)
    a3.set_xlabel("false positive rate"); a3.set_ylabel("true positive rate")
    a3.set_title("ROC: the marker is your cutoff", fontsize=10); a3.legend(fontsize=8)
    plt.tight_layout(); plt.show()

interact(cutoff_explorer,
         cutoff=FloatSlider(0.5, min=0.02, max=0.98, step=0.02, continuous_update=False));


In [ ]:
# --- What the classifier learned ---------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))

vv = np.linspace(0.08, 0.62, 220)
dd = np.linspace(7, 13, 120)
VV, DD = np.meshgrid(vv, dd)
P = clf.predict_proba(np.c_[VV.ravel(), DD.ravel()])[:, 1].reshape(VV.shape)

cs = a1.contourf(VV, DD, P, levels=20, cmap="RdYlGn", alpha=0.8)
a1.contour(VV, DD, P, levels=[0.5], colors="k", linewidths=2.2)
jit = clean.diameter + np.random.uniform(-0.35, 0.35, len(clean))
a1.scatter(clean.vf, jit, c=clean.passes, cmap="RdYlGn", s=7,
           edgecolors="k", linewidths=0.15, alpha=0.75)
a1.set_xlabel("$V_f$"); a1.set_ylabel("diameter (jittered)")
a1.set_title("the decision boundary is almost vertical", fontsize=10); a1.grid(False)
plt.colorbar(cs, ax=a1, shrink=0.85, label="P(pass)")

xs = np.linspace(0.08, 0.62, 300)
probs = clf.predict_proba(np.c_[xs, np.full_like(xs, 10)])[:, 1]
a2.plot(xs, probs, color=C_DATA, lw=2.5)
a2.axhline(0.5, color="k", ls="--", lw=1)
v50 = xs[np.argmin(np.abs(probs - 0.5))]
a2.axvline(v50, color=C_FIT, ls="--", lw=1.5)
a2.scatter(clean.vf, clean.passes, s=6, alpha=0.15, color=C_BAD)
a2.set_xlabel("$V_f$"); a2.set_ylabel("P(pass)")
a2.set_title(f"the sigmoid, at d=10\n50% confidence at $V_f$ = {v50:.3f}", fontsize=10)
plt.tight_layout(); plt.show()

shift = -clf.coef_[0][1] / clf.coef_[0][0] * 4     # boundary movement over d = 8 to 12
print(f"coefficients:  Vf {clf.coef_[0][0]:+.3f}   diameter {clf.coef_[0][1]:+.4f}")
print(f"Across the whole diameter range the boundary moves by {abs(shift):.3f} in volume fraction,")
print(f"against a spacing of 0.10 between adjacent volume fraction levels in the dataset.")


---

# What to take away

1. Two numbers and two parameters predict the average transverse stiffness of this composite with
   $R^2 = 0.95$. Simple models are often most of the answer, and they should be the first thing you
   try, not the thing you skip.
2. The fitted coefficients are readable as physics. The slope is stiffness gained per unit fibre.
   The diameter coefficient is about $-0.016$ GPa per unit diameter, smaller than the scatter
   between realisations and worth 0.00009 of $R^2$: indistinguishable from the zero that
   homogenisation predicts.
3. Residual plots, not $R^2$, tell you what a model is missing.
4. Anisotropy is invisible to composition. $R^2 = 0.0005$ from volume fraction, $V_f^2$ and
   diameter together. The information was discarded by the choice of input representation, before
   any model was fitted.
5. Logistic regression is a linear model followed by a squashing function, which is one neuron.

---

# Exercises

### 1. Read a coefficient as physics
Fit `G23` against $V_f$. What are the slope and intercept, and what does each one mean physically?
What should the intercept be, and is it?

### 2. Predict before you run
The dataset contains `v23`, the transverse Poisson's ratio. Write down whether you expect it to
increase or decrease with volume fraction, and why, **then** fit it and check. If you were wrong,
work out why.

### 3. Find where the linear model breaks
`sy_mean` is the average yield stress. Fit it against $V_f$ linearly, then with $V_f^2$. Compare
$R^2$ against the same two fits for `E_mean`. Which property is harder, and what does the residual
plot suggest is going on?

### 4. The null feature is not entirely null
Part 2 showed that diameter does not shift the **mean** stiffness. Now group the data by diameter
and compute the **standard deviation** of `dE` within each group. What do you find, and why?

*Hint: the simulation domain is the same size for every microstructure. At a fixed volume fraction,
how many fibres fit in it when the fibres are larger, and what does that do to how representative a
single realisation is?*

### 5. Set the cutoff like an engineer
You are screening microstructures for a part where a false pass is ten times more costly than a
false fail. Using the test set probabilities, find the cutoff that minimises total cost. How far is
it from 0.5?

### 6. Break the model honestly
Train on volume fractions 10 to 40% only, then predict at 50 and 60%. Report the error. Do the same
with the quadratic model. Which extrapolates better, and would you trust either?
